# Prototypes and Scraping experiments for ARIA
Testing out API calls to OpenAlex and Wikipedia, and trying out reportlab to see why the footers are broken.

In [1]:
import requests
import xml.etree.ElementTree as ET

In [2]:
# Testing Wikipedia search API
url = "https://en.wikipedia.org/w/api.php"
params = {
    "action": "query",
    "format": "json",
    "generator": "search",
    "gsrsearch": "NVIDIA Blackwell GPU supply chain",
    "gsrlimit": 2,
    "prop": "extracts|info",
    "exintro": 1,
    "explaintext": 1,
    "inprop": "url",
}
res = requests.get(url, params=params)
# print(res.json().keys())
pages = res.json().get("query", {}).get("pages", {})
for p in pages.values():
    print(p.get("title"))

In [3]:
# Testing arXiv API parsing
# I spent 30 mins figuring out why ElementTree query wasn't returning entries.
# Namespace was the issue! Need to pass namespace dict.
query = "self-correcting RAG"
url = "https://export.arxiv.org/api/query"
params = {
    "search_query": query,
    "max_results": 2
}
r = requests.get(url, params=params)
root = ET.fromstring(r.content)

# FAILED ATTEMPT:
entries_failed = root.findall("entry")
print(f"Found {len(entries_failed)} entries (failed without ns)")

# FIXED ATTEMPT:
ns = {"atom": "http://www.w3.org/2005/Atom"}
entries = root.findall("atom:entry", ns)
print(f"Found {len(entries)} entries (with ns)")
for entry in entries[:1]:
    print("Title:", entry.find("atom:title", ns).text.replace('\n', ' ').strip())

Found 0 entries (failed without ns)
Found 2 entries (with ns)
Title: Self-correcting RAG for agentic workflows


## ReportLab PDF page numbering experiment
Why is it so hard to get page numbers in ReportLab?
If I just run `doc.build(story)`, the footer gets drawn during page construction, but the total page count isn't known yet.
Let's test subclassing `canvas.Canvas`.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4

class TestCanvas(canvas.Canvas):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pages = []
        
    def showPage(self):
        self.pages.append(dict(self.__dict__))
        self._startPage()
        
    def save(self):
        # Check if the page count works
        print(f"Total pages rendered: {len(self.pages)}")
        super().save()

In [5]:
# Quick check: does yfinance work?
try:
    import yfinance as yf
    msft = yf.Ticker("MSFT")
    hist = msft.history(period="1mo")
    if not hist.empty:
        print("yfinance is installed! MSFT data:")
        print(f"Latest Close: {hist.iloc[-1]['Close']:.2f}")
    else:
        print("yfinance returned empty dataset.")
except Exception as e:
    print("yfinance not installed or API error:", e)

yfinance is installed! MSFT data:
Latest Close: 418.50
